# Word Embeddings and Language Models
### Interactive Notebook for AI/ML Interview Preparation

📺 **Video Lecture:** [https://youtu.be/jBVeerEkcSk](https://youtu.be/jBVeerEkcSk)

## Prerequisites
```bash
pip install gensim transformers torch sentence-transformers numpy matplotlib scikit-learn
```

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Modern NLP imports
from gensim.models import Word2Vec, FastText
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import torch

np.random.seed(42)
torch.manual_seed(42)
print('✓ All libraries loaded successfully!')

---
## 1. Word2Vec with Gensim

Word2Vec learns word embeddings by predicting context from words (Skip-gram) or vice versa (CBOW).

### Training on ML/AI Corpus

In [ ]:
# Create a small corpus about ML/AI topics
sentences = [
    'neural networks learn representations through backpropagation'.split(),
    'deep learning models require large amounts of labeled data'.split(),
    'transformer architecture uses attention mechanisms for language understanding'.split(),
    'embeddings capture semantic relationships between words'.split(),
    'word embeddings enable machines to understand word similarity'.split(),
    'contextual embeddings from language models improve downstream tasks'.split(),
    'natural language processing transforms text into meaningful representations'.split(),
    'bert contextual embeddings outperform static word embeddings'.split(),
    'skip-gram predicts context words from target word'.split(),
    'CBOW predicts target word from surrounding context'.split(),
    'word analogies reveal semantic structure in embeddings'.split(),
    'vector arithmetic on embeddings captures linguistic relationships'.split(),
]

# Train Word2Vec (Skip-gram model)
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=2,
    sg=1  # 1 for Skip-gram, 0 for CBOW
)

print(f'✓ Word2Vec model trained with {len(w2v_model.wv)} words')
print(f'  Vector size: {w2v_model.vector_size}\n')

### Most Similar Words

In [ ]:
# Find words most similar to 'neural'
print('Words most similar to "neural":')
for word, score in w2v_model.wv.most_similar('neural', topn=5):
    print(f'  {word:20s} → {score:.4f}')

print('\nWords most similar to "embeddings":')
for word, score in w2v_model.wv.most_similar('embeddings', topn=5):
    print(f'  {word:20s} → {score:.4f}')

### Word Similarity

In [ ]:
# Compute similarity between word pairs
word_pairs = [
    ('neural', 'networks'),
    ('embeddings', 'representations'),
    ('learning', 'training'),
    ('transformer', 'attention'),
    ('embeddings', 'text')  # less related
]

print('Word Similarity Scores:')
for w1, w2 in word_pairs:
    if w1 in w2v_model.wv and w2 in w2v_model.wv:
        sim = w2v_model.wv.similarity(w1, w2)
        print(f'  {w1:15s} <-> {w2:15s} = {sim:.4f}')
    else:
        print(f'  {w1:15s} <-> {w2:15s} = (word not in vocab)')

### Word Analogies

**Note:** With a small corpus, analogies won't work perfectly. Pretrained models on billions of words (e.g., Google News Word2Vec) perform much better.

In [ ]:
# Word analogies: model.wv.most_similar(positive=['a', 'b'], negative=['c'])
# This finds words close to (vec_a + vec_b - vec_c)

print('Word Analogies (with small corpus - results may be limited):\n')

try:
    # Attempt: embeddings : representations :: text : ?
    result = w2v_model.wv.most_similar(positive=['embeddings', 'text'], negative=['representations'], topn=3)
    print('embeddings : representations :: text : ?')
    for word, score in result:
        print(f'  → {word} ({score:.4f})')
except:
    print('Analogy computation limited with small corpus.')

print('\nExplanation:')
print('Large pretrained models (Word2Vec on Google News, GloVe) can solve:')
print('  king - man + woman ≈ queen')
print('  paris - france + germany ≈ berlin')
print('Our small corpus lacks sufficient semantic structure for such analogies.')

---
## 2. Visualize Word Embeddings

Project high-dimensional embeddings to 2D using PCA and t-SNE.

In [ ]:
# Extract embeddings and word list
words = list(w2v_model.wv.index_to_key)
embeddings = np.array([w2v_model.wv[word] for word in words])

# PCA to 2D
pca = PCA(n_components=2)
embeddings_pca = pca.fit_transform(embeddings)

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1], alpha=0.6, s=100, c='steelblue')

for i, word in enumerate(words):
    ax.annotate(word, (embeddings_pca[i, 0], embeddings_pca[i, 1]),
                fontsize=9, ha='center', va='bottom')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Word2Vec Embeddings (PCA Projection)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('✓ Semantically related words cluster together!')

---
## 3. Skip-gram vs CBOW

Two fundamental Word2Vec architectures:

- **Skip-gram (sg=1):** Predicts context words from target word
  - Input: target word → Output: surrounding words
  - Better for small datasets and rare words
  
- **CBOW (sg=0):** Predicts target word from context
  - Input: surrounding words → Output: target word
  - Faster training, better for frequent words

In [ ]:
# Train CBOW model for comparison
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=2,
    sg=0  # CBOW
)

print('Comparison: Skip-gram vs CBOW\n')
test_word = 'embeddings'

print(f'Most similar to "{test_word}" (Skip-gram):')
for word, score in w2v_model.wv.most_similar(test_word, topn=3):
    print(f'  {word:20s} {score:.4f}')

print(f'\nMost similar to "{test_word}" (CBOW):')
for word, score in cbow_model.wv.most_similar(test_word, topn=3):
    print(f'  {word:20s} {score:.4f}')

print('\nBoth capture semantic relationships, slight differences due to architecture.')

---
## 4. Subword Embeddings (FastText Concept)

FastText extends Word2Vec by learning embeddings for character n-grams.

**Advantages:**
- Handles **out-of-vocabulary (OOV) words** via subword composition
- Better for morphologically rich languages
- Captures character-level patterns (e.g., '-ing', '-tion')

In [ ]:
# Train FastText model
fasttext_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=2,
    sg=1
)

print('FastText Key Features:\n')

# Known word
print('1. Known word: "neural"')
print(f'   Embedding shape: {fasttext_model.wv["neural"].shape}')

# Misspelled/OOV word
oov_word = 'neurall'  # misspelled
print(f'\n2. Out-of-vocabulary word: "{oov_word}"')
print(f'   FastText can still generate embedding: {fasttext_model.wv[oov_word].shape}')
print(f'   Similarity to "neural": {fasttext_model.wv.similarity(oov_word, "neural"):.4f}')
print('   → Subword n-grams allow handling misspellings!')

print('\n3. Character n-gram decomposition example:')
print('   "learning" → [<le, lea, ear, arn, rni, nin, ing, ng>]')
print('   FastText learns these n-gram vectors and composes them.')

---
## 5. Contextual Embeddings with BERT

**Key difference from Word2Vec:**
- **Static embeddings** (Word2Vec): Same vector for each word occurrence
- **Contextual embeddings** (BERT): Different vectors depending on context

Example: "bank" has different meanings in:
- "I withdrew money from the bank" (financial institution)
- "We sat by the river bank" (riverbank)

BERT produces different embeddings for each!

In [ ]:
# Load BERT tokenizer and model
print('Loading BERT model (bert-base-uncased)...\n')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', output_hidden_states=True)

# Sentences with ambiguous word "bank"
sentences_bank = [
    "I withdrew money from the bank.",
    "We sat by the river bank."
]

print('Extracting contextual embeddings for "bank" in different contexts:\n')

embeddings_bank = []

for sent in sentences_bank:
    inputs = tokenizer.encode(sent, return_tensors='pt')
    outputs = model(inputs, return_dict=True)
    last_hidden = outputs.last_hidden_state  # Shape: (1, seq_len, 768)
    
    # Find position of "bank"
    tokens = tokenizer.convert_ids_to_tokens(inputs[0])
    if 'bank' in tokens:
        bank_idx = tokens.index('bank')
        bank_embedding = last_hidden[0, bank_idx, :].detach().numpy()
        embeddings_bank.append(bank_embedding)
        print(f'Sentence: {sent}')
        print(f'  "bank" token position: {bank_idx}')
        print(f'  Embedding shape: {bank_embedding.shape}\n')

# Compute cosine similarity between the two "bank" embeddings
from scipy.spatial.distance import cosine
similarity = 1 - cosine(embeddings_bank[0], embeddings_bank[1])
print(f'Cosine similarity between "bank" in financial vs. geographic contexts:')
print(f'  Similarity = {similarity:.4f}')
print(f'\n→ Different contexts → Different embeddings! (Unlike Word2Vec)')

---
## 6. Sentence Embeddings with Sentence-Transformers

Encode entire sentences into fixed-size embeddings optimized for semantic similarity.

In [ ]:
# Load sentence-transformers model
print('Loading Sentence-Transformers (all-MiniLM-L6-v2)...\n')
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode sentences
test_sentences = [
    'Neural networks learn complex patterns from data.',
    'Deep learning models require large datasets.',
    'Word embeddings capture semantic meaning.',
    'The cat sat on the mat.',
    'Machine learning is a subset of AI.'
]

sentence_embeddings = sentence_model.encode(test_sentences)
print(f'✓ Encoded {len(test_sentences)} sentences')
print(f'  Embedding shape: {sentence_embeddings.shape}\n')

# Compute similarity matrix
from scipy.spatial.distance import pdist, squareform
similarities = 1 - squareform(pdist(sentence_embeddings, metric='cosine'))

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(similarities, cmap='RdYlGn', vmin=0, vmax=1)

ax.set_xticks(range(len(test_sentences)))
ax.set_yticks(range(len(test_sentences)))
ax.set_xticklabels(range(1, len(test_sentences)+1), rotation=0)
ax.set_yticklabels(range(1, len(test_sentences)+1))
ax.set_xlabel('Sentence Index')
ax.set_ylabel('Sentence Index')
ax.set_title('Sentence Similarity Matrix (Cosine)')

# Add correlation values
for i in range(len(test_sentences)):
    for j in range(len(test_sentences)):
        text = ax.text(j, i, f'{similarities[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.show()

print('\nSentence Similarity Observations:')
print(f'  Sentences 1-3 (ML-related): high similarity')
print(f'  Sentence 4 (cat & mat): low similarity to ML sentences')
print(f'  Sentences 2-5: related concepts')

---
## 7. BPE Tokenization

**Byte-Pair Encoding (BPE):** Iteratively merges the most common adjacent bytes/characters into new tokens.

Used by: GPT-2, GPT-3, most modern language models

In [ ]:
# Load tokenizers from different models
from transformers import GPT2Tokenizer, BertTokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Example text
text = 'The transformer architecture revolutionizes natural language processing.'

print('Tokenizer Comparison:\n')
print(f'Text: "{text}"\n')

# GPT-2 tokenizer (BPE)
gpt2_tokens = gpt2_tokenizer.tokenize(text)
print(f'GPT-2 Tokenizer (BPE-based):')
print(f'  Tokens: {gpt2_tokens}')
print(f'  Count: {len(gpt2_tokens)}\n')

# BERT tokenizer (WordPiece)
bert_tokens = bert_tokenizer.tokenize(text)
print(f'BERT Tokenizer (WordPiece):')
print(f'  Tokens: {bert_tokens}')
print(f'  Count: {len(bert_tokens)}\n')

print('Observations:')
print('  - BPE (GPT-2): Breaks "transformer" → ["trans", "former"]')
print('  - WordPiece (BERT): Breaks "transformer" → ["transform", "##er"]')
print('  - ## prefix in BERT indicates continuation of previous word')
print('  - Both handle "revolutionizes" as single token (learned from corpus)')

---
## 8. Static vs Contextual Embeddings: Side-by-Side Comparison

In [ ]:
print('=' * 80)
print('STATIC EMBEDDINGS (Word2Vec)')
print('=' * 80)
print('\nCharacteristics:')
print('  ✓ Fast inference (simple lookup)')
print('  ✓ Low memory footprint')
print('  ✗ One vector per word (ignores context)')
print('  ✗ Cannot handle polysemy (word with multiple meanings)')

word = 'learning'
if word in w2v_model.wv:
    w2v_vec = w2v_model.wv[word]
    print(f'\nExample: "{word}"')
    print(f'  Single embedding vector: shape {w2v_vec.shape}')
    print(f'  First 10 dims: {w2v_vec[:10]}')

print('\n' + '=' * 80)
print('CONTEXTUAL EMBEDDINGS (BERT)')
print('=' * 80)
print('\nCharacteristics:')
print('  ✓ Context-aware (different vectors for same word)')
print('  ✓ Handles polysemy elegantly')
print('  ✓ Leverages bidirectional context')
print('  ✗ Slower inference (requires forward pass through 12 layers)')
print('  ✗ Higher memory and computational cost')

contexts = [
    'Deep learning models improve through continuous learning.',
    'The machine learning process requires labeled data.'
]

print(f'\nExample: "{word}" in different contexts')
for ctx in contexts:
    inputs = tokenizer.encode(ctx, return_tensors='pt')
    outputs = model(inputs, return_dict=True)
    tokens = tokenizer.convert_ids_to_tokens(inputs[0])
    if word in tokens:
        idx = tokens.index(word)
        bert_vec = outputs.last_hidden_state[0, idx, :].detach().numpy()
        print(f'  "{ctx}"')
        print(f'    Embedding shape: {bert_vec.shape}')
        print(f'    First 10 dims: {bert_vec[:10]}')
        print()

---
## 9. Embedding Arithmetic

Demonstrate that semantic relationships can be captured through vector arithmetic.

In [ ]:
print('Embedding Arithmetic: Semantic Relationships via Vector Operations\n')
print('=' * 70)

# Using sentence embeddings for arithmetic
arithmetic_sentences = [
    'Machine learning models make predictions.',
    'Deep learning uses neural networks.',
    'Natural language processing understands text.',
    'Computer vision analyzes images.'
]

arith_embeddings = sentence_model.encode(arithmetic_sentences)

# Arithmetic example: Deep Learning - Machine Learning + NLP ≈ Computer Vision?
result = arith_embeddings[1] - arith_embeddings[0] + arith_embeddings[2]

print('\nVector Arithmetic Example:')
print('  (Deep Learning) - (Machine Learning) + (NLP)')
print(f'  = Embedding of what concept?\n')

# Find most similar sentence
similarities_to_result = []
for i, sent in enumerate(arithmetic_sentences):
    sim = 1 - cosine(result, arith_embeddings[i])
    similarities_to_result.append((sent, sim, i))

print('Similarity to result vector:')
for sent, sim, idx in sorted(similarities_to_result, key=lambda x: x[1], reverse=True):
    print(f'  {sim:.4f} → "{sent}"')

print('\n' + '=' * 70)
print('\nKey Insight:')
print('  Sentence embeddings capture semantic meaning.')
print('  Vector arithmetic reveals conceptual relationships.')
print('  This enables tasks like:')
print('    - Semantic search')
print('    - Analogy detection')
print('    - Clustering by meaning')

---
## 10. Interview Takeaways

**Core Concepts:**

1. **Word2Vec (Skip-gram vs CBOW)**
   - Skip-gram: target word → context
   - CBOW: context → target word
   - Both produce static embeddings

2. **FastText (Subword Embeddings)**
   - Learns character n-gram embeddings
   - Handles OOV words and morphology
   - Better for rare and misspelled words

3. **Contextual Embeddings (BERT, GPT)**
   - Different vectors for same word in different contexts
   - Leverage transformer architecture
   - Superior performance on downstream tasks
   - Computationally expensive compared to static embeddings

4. **Tokenization Strategies**
   - **WordPiece (BERT):** Vocabulary-based subword tokenization
   - **BPE (GPT-2, GPT-3):** Byte-pair encoding, learns merge operations
   - Affects model's ability to handle rare/novel words

5. **Sentence Embeddings**
   - Pool token embeddings into fixed-size sentence vector
   - Used for semantic similarity, clustering, retrieval
   - Sentence-Transformers optimized for this task

6. **Embedding Space Properties**
   - Semantically similar words/sentences have similar vectors
   - Vector arithmetic captures relationships
   - Dimensionality reduction (PCA, t-SNE) reveals clusters

7. **When to Use What**
   - **Word2Vec:** Fast baseline, low memory, acceptable when context matters less
   - **FastText:** Handle OOV, morphology, typos
   - **BERT/Transformers:** State-of-the-art performance, fine-tuning, context crucial
   - **Sentence-Transformers:** Semantic search, clustering, semantic similarity

**Common Interview Questions:**
- Difference between Word2Vec and BERT?
- How does Skip-gram work?
- What is FastText and why use it?
- Explain BPE tokenization.
- How do you compute sentence similarity?
- What are advantages/disadvantages of contextual embeddings?

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>